## Car Troubleshooting System  (forward)

In [7]:
car_rules = [
    ({"engine_cranks_no_start"}, "battery_dead", "Engine cranks but does not start -> battery could be weak or dead."),
    ({"engine_does_not_crank", "lights_dim_or_no_lights"}, "battery_dead", "Engine doesn't crank and lights are dim -> battery likely dead."),
    ({"battery_ok", "fuel_low"}, "no_fuel", "Battery is OK but there's no fuel -> check fuel level or fuel pump."),
    ({"battery_ok", "fuel_ok", "spark_missing"}, "ignition_problem", "Battery and fuel OK but no spark -> ignition/coil/plug issue."),
    ({"starter_clicks", "engine_does_not_crank"}, "starter_fault", "Starter clicks but engine does not crank -> starter motor or solenoid issue."),
    ({"engine_cranks_no_start", "fuel_ok", "spark_ok"}, "engine_mechanical_issue", "Cranks, has fuel and spark -> possible mechanical issue (timing, compression).")
]

In [8]:
def apply_forward_rules(facts, rules):
    derived = set(facts)
    trace = []
    changed = True
    while changed:
        changed = False
        for conds, concl, expl in rules:
            if conds.issubset(derived) and concl not in derived:
                derived.add(concl)
                trace.append(f"Applied rule -> IF {' AND '.join(conds)} THEN {concl}. Explanation: {expl}")
                changed = True
    return derived, trace
    

In [9]:
def get_yes_no(question):
    while True:
        ans = input(question + " (yes/no): ").strip().lower()
        if ans in ("yes","y"):
            return True
        if ans in ("no","n"):
            return False
        print("Please answer 'yes' or 'no'.")
        

In [10]:
def run_car_forward():
    print("=== Car Troubleshooting (Forward Chaining) ===")

    facts = set()

    if get_yes_no("Does the engine crank (turn) when you turn the key?"):
        facts.add("engine_cranks")
        if get_yes_no("Does the engine crank but not start?"):
            facts.add("engine_cranks_no_start")
    else:
        facts.add("engine_does_not_crank")
    if get_yes_no("Do the dashboard lights appear dim or not light up?"):
        facts.add("lights_dim_or_no_lights")
    else:
        facts.add("lights_normal")
    if get_yes_no("Do you hear a single click when you try to start?"):
        facts.add("starter_clicks")
    if get_yes_no("Have you checked the battery (or is the battery recently replaced)?"):
        if get_yes_no("Is the battery okay (voltage above ~12V)?"):
            facts.add("battery_ok")
        else:
            facts.add("battery_bad")
    else:

        pass
    if get_yes_no("Is there fuel in the tank?"):
        facts.add("fuel_ok")
    else:
        facts.add("fuel_low")
    if get_yes_no("Do you hear the fuel pump prime when you turn the key?"):
        facts.add("fuel_pump_primes")
    if get_yes_no("Have you checked for spark at the plugs?"):
        if get_yes_no("Is spark present?"):
            facts.add("spark_ok")
        else:
            facts.add("spark_missing")

    print("\nCollected facts:", facts)
    derived, trace = apply_forward_rules(facts, car_rules)
    print("\nReasoning steps:")
    if trace:
        for step in trace:
            print("-", step)
    else:
        print("- No rules could be applied with the given facts.")

    conclusions = [f for f in derived if f in {r[1] for r in car_rules}]
    if conclusions:
        print("\nConclusions / Diagnoses:")
        for c in conclusions:
            if c == "battery_dead" and ("battery_bad" in facts or "lights_dim_or_no_lights" in facts):
                print("- Likely battery problem (battery_dead).")
            elif c == "no_fuel":
                print("- Likely no fuel / fuel system issue (no_fuel).")
            elif c == "ignition_problem":
                print("- Likely ignition problem (ignition_problem).")
            elif c == "starter_fault":
                print("- Likely starter motor/solenoid issue (starter_fault).")
            elif c == "engine_mechanical_issue":
                print("- Possible mechanical engine issue (engine_mechanical_issue).")
            else:
                print(f"- {c}")
    else:
        print("\nNo clear diagnosis could be reached. Consider checking battery, fuel, and ignition components manually.")




##  Animal Identification System (backward)

In [11]:
animal_rules = [
    ({"lays_eggs", "has_feathers"}, "bird", "Lays eggs and has feathers -> bird."),
    ({"gives_milk", "has_fur"}, "mammal", "Gives milk and has fur -> mammal."),
    ({"has_fins", "lives_in_water"}, "fish", "Has fins and lives in water -> fish."),
    ({"has_scales", "lives_in_water"}, "fish", "Has scales and lives in water -> fish."),
    ({"lays_eggs", "is_aquatic", "has_scales"}, "reptile_or_fish", "Lays eggs and is aquatic and has scales -> likely fish or amphibious reptile."),
    ({"has_shell", "lays_eggs"}, "reptile", "Has shell and lays eggs -> reptile."),
    ({"flies", "has_feathers"}, "bird", "Flies and has feathers -> bird."),
    ({"is_domestic", "gives_milk"}, "cow", "Domestic and gives milk -> cow (a mammal).")
]


In [12]:
def backward_chain(goal, known_facts, rules, asked_questions=None, trace=None):

    if asked_questions is None:
        asked_questions = set()
    if trace is None:
        trace = []

    if goal in known_facts:
        trace.append(f"Goal '{goal}' already in known facts.")
        return True, known_facts, trace
    
    relevant = [r for r in rules if r[1] == goal]
    if not relevant:
        
        if goal not in asked_questions:
 
            ans = get_yes_no(f"Does the animal have the property '{goal.replace('_',' ')}'?")
            asked_questions.add(goal)
            if ans:
                known_facts.add(goal)
                trace.append(f"User confirmed fact: {goal}.")
                return True, known_facts, trace
            else:
                trace.append(f"User denied fact: {goal}.")
                return False, known_facts, trace
        else:
            return False, known_facts, trace
    
    for conds, concl, expl in relevant:
        trace.append(f"Considering rule: IF {' AND '.join(conds)} THEN {concl}. ({expl})")
        all_proved = True
        for c in conds:
            if c in known_facts:
                trace.append(f" - Fact '{c}' already known.")
                continue
            
            proved, known_facts, trace = backward_chain(c, known_facts, rules, asked_questions, trace)
            if not proved:
                trace.append(f" - Could not prove condition '{c}' for rule concluding '{concl}'.")
                all_proved = False
                break
        if all_proved:
            known_facts.add(concl)
            trace.append(f"Rule succeeded. Concluded: {concl}.")
            return True, known_facts, trace
    
    if goal not in asked_questions:
        ans = get_yes_no(f"As a fallback: Does the animal have the property '{goal.replace('_',' ')}'?")
        asked_questions.add(goal)
        if ans:
            known_facts.add(goal)
            trace.append(f"User confirmed fallback fact: {goal}.")
            return True, known_facts, trace
        else:
            trace.append(f"User denied fallback fact: {goal}.")
            return False, known_facts, trace
    return False, known_facts, trace

    

## Program

In [13]:
def main_menu():
    print("Expert Systems - Task 2\nChoose an option:")
    print("1. Run Car Troubleshooting (Forward Chaining)")
    print("2. Run Animal Identification (Backward Chaining)")
    print("3. Exit")
    while True:
        choice = input("Select 1/2/3: ").strip()
        if choice == "1":
            run_car_forward()
            break
        elif choice == "2":
            run_animal_backward()
            break
        elif choice == "3":
            print('Exiting.')
            break
        else:
            print("Invalid choice. Please enter 1, 2, or 3.")


In [16]:
main_menu()

Expert Systems - Task 2
Choose an option:
1. Run Car Troubleshooting (Forward Chaining)
2. Run Animal Identification (Backward Chaining)
3. Exit


Select 1/2/3:  1


=== Car Troubleshooting (Forward Chaining) ===


Does the engine crank (turn) when you turn the key? (yes/no):  no
Do the dashboard lights appear dim or not light up? (yes/no):  yes
Do you hear a single click when you try to start? (yes/no):  yes
Have you checked the battery (or is the battery recently replaced)? (yes/no):  no
Is there fuel in the tank? (yes/no):  yes
Do you hear the fuel pump prime when you turn the key? (yes/no):  no
Have you checked for spark at the plugs? (yes/no):  no



Collected facts: {'engine_does_not_crank', 'fuel_ok', 'starter_clicks', 'lights_dim_or_no_lights'}

Reasoning steps:
- Applied rule -> IF engine_does_not_crank AND lights_dim_or_no_lights THEN battery_dead. Explanation: Engine doesn't crank and lights are dim -> battery likely dead.
- Applied rule -> IF engine_does_not_crank AND starter_clicks THEN starter_fault. Explanation: Starter clicks but engine does not crank -> starter motor or solenoid issue.

Conclusions / Diagnoses:
- Likely starter motor/solenoid issue (starter_fault).
- Likely battery problem (battery_dead).
